# 02 · Dataset maestro

**Proyecto:** LINE — Auditor Médico Digital · Health & Life IPS SAS

Fusiona las 5 tablas limpias en un único dataset de modelado, con **control
explícito de data leakage** y **features alineadas con reglas de negocio de
facturación**.

| Entrada | Salida |
|---|---|
| `data/processed/*_clean.csv` (5 tablas) | `data/processed/dataset_maestro.csv` |
| | `outputs/reports/columnas_modelado.json` |

**Control de leakage.** El target (`target` / `resultado`) fue generado por un
sistema de reglas previo. Sus salidas directas (`tipo_alerta`, `severidad`,
`descripcion_alerta`) **nunca** se usan como features — solo quedan en el
maestro para análisis explicativo. Las features de modelado son únicamente las
disponibles *en producción, antes de auditar*.

**Lógica corregida (fix de esta versión).** La versión anterior de este
notebook intentaba agrupar por atención pero rompía el contrato de salida:
colapsaba a 1 fila = 1 atención y perdía columnas de grano fino
(`codigo_cups`, `codigo_cups_facturado`, `cantidad_realizada`,
`cantidad_facturada`, `valor_unitario`, `valor_total`, `tipo_item`,
`soporte_clinico`, `profesional_responsable`, …) que **05, 07 y 08 necesitan
tal cual**. Además, dos celdas quedaron tipeadas como *Markdown* en vez de
*Code*, así que nunca se ejecutaban.

Esta versión mantiene el **grano de `cruce`** (1 fila = 1 evaluación
HC-vs-facturación, 3.126 filas, igual que antes) — eso es lo que permite que
05/07/08 sigan funcionando sin tocarles una línea — pero corrige la forma en
que se calculan las features de comparación:

1. Agrupar `hc_detalle` por `id_atencion` → set de códigos CUPS con soporte clínico.
2. Agrupar `prefactura` por `id_atencion` → set de códigos CUPS facturados.
3. Comparar los dos sets **dentro de cada atención** (no solo la fila pareada
   de `cruce`, que es apenas *una* observación dentro de ese set).
4. Traer `tipo_atencion` y diagnóstico de `atenciones` para las reglas de
   negocio (ambulatorio/hospitalario/urgencia, alto valor, etc.).

`codigo_facturado_tiene_soporte_en_atencion` deja de ser "¿esta fila
específica coincide?" y pasa a ser "¿este código facturado aparece en el set
de códigos con soporte de *esa misma atención*?" — la comparación fila a fila
que hacen 07/08 (`codigo_cups_facturado == codigo_cups` en la misma fila)
sigue siendo válida porque las FK de `cruce` (`id_detalle_hc`,
`id_prefactura`) sí son la pareja correcta (verificado más abajo: donde
`tipo_alerta == CODIGO_NO_COINCIDE` los códigos difieren 120/120 veces; en
el resto coinciden). Lo que faltaba era la señal *adicional* a nivel de
conjunto, que aquí se agrega como columnas nuevas sin romper nada.

## 0 · Configuración (celda autocontenida)
Misma convención de rutas que el resto del pipeline (01-08): detecta la raíz
del proyecto, crea carpetas de salida y define utilidades de escritura
robusta (atómica, con reintentos, por si la carpeta vive bajo OneDrive).

In [3]:
from pathlib import Path
import json
import os
import time

import numpy as np
import pandas as pd

try:  # ejecutando como script .py desde src/
    ROOT = Path(__file__).resolve().parents[1]
except NameError:  # ejecutando como notebook desde notebooks/
    ROOT = Path.cwd()
    if ROOT.name in ("notebooks", "src"):
        ROOT = ROOT.parent

DATA_PROC = ROOT / "data" / "processed"
OUT_REP = ROOT / "outputs" / "reports"
for _d in (DATA_PROC, OUT_REP):
    _d.mkdir(parents=True, exist_ok=True)

try:
    display  # noqa: B018 — definido por IPython en notebooks
except NameError:
    display = print


def reintentar(fn, intentos=6, espera=0.5):
    """Reintenta fn() ante OSError (bloqueos transitorios del sync de OneDrive)."""
    for _i in range(intentos):
        try:
            return fn()
        except OSError:
            if _i == intentos - 1:
                raise
            time.sleep(espera * (2 ** _i))


def to_csv_seguro(df, path, **kw):
    path = Path(path)
    tmp = path.with_suffix(path.suffix + ".tmp")
    reintentar(lambda: df.to_csv(tmp, **kw))
    reintentar(lambda: os.replace(tmp, path))


def write_text_seguro(path, texto, encoding="utf-8"):
    path = Path(path)
    tmp = path.with_suffix(path.suffix + ".tmp")
    reintentar(lambda: tmp.write_text(texto, encoding=encoding))
    reintentar(lambda: os.replace(tmp, path))


ENTRADAS = [DATA_PROC / f"{n}_clean.csv" for n in
            ("cruce", "prefactura", "hc_detalle", "atenciones", "pacientes")]
_faltan = [str(p_) for p_ in ENTRADAS if not p_.exists()]
assert not _faltan, (
    "FALTAN INSUMOS:\n  - " + "\n  - ".join(_faltan)
    + "\n→ Ejecuta primero el notebook 01_limpieza.ipynb."
)

cruce = pd.read_csv(DATA_PROC / "cruce_clean.csv")
prefactura = pd.read_csv(DATA_PROC / "prefactura_clean.csv")
hc = pd.read_csv(DATA_PROC / "hc_detalle_clean.csv")
atenciones = pd.read_csv(DATA_PROC / "atenciones_clean.csv")
pacientes = pd.read_csv(DATA_PROC / "pacientes_clean.csv")

for df, cols in [(prefactura, ["fecha_facturacion"]),
                  (hc, ["fecha_registro"]),
                  (atenciones, ["fecha_atencion"])]:
    for col in cols:
        df[col] = pd.to_datetime(df[col])

print(f"✅ Insumos OK: {len(cruce):,} cruce, {len(prefactura):,} prefactura, "
      f"{len(hc):,} hc, {len(atenciones):,} atenciones, {len(pacientes):,} pacientes")

✅ Insumos OK: 3,126 cruce, 2,974 prefactura, 3,058 hc, 1,200 atenciones, 300 pacientes


## 1 · Sets de CUPS por atención (paso 1-2 de la lógica corregida)
Se agrupa **antes** de tocar el grano de `cruce`, para tener disponible el
set completo de códigos de cada atención al momento de evaluar cada fila.

> **Nota (hallazgo de auditoría):** la salida reporta **1.201** atenciones y no
> 1.200 porque `hc_detalle` trae 2 filas de la atención **`ATN-JEF-000001`**,
> que no existe en `atenciones` (ver cierre del notebook 01). Esa atención
> fantasma **no entra al dataset maestro** — `cruce` nunca la referencia — pero
> sí aparece en estos conteos globales y explica los 2 códigos `SOLO_HC`
> (`890205`, `902201`) del notebook 03.

In [5]:
# 1. Agrupar hc_detalle por id_atencion → set de códigos CUPS con soporte clínico
hc_por_atencion = hc.groupby("id_atencion").agg(
    cups_hc_set=("codigo_cups", lambda x: frozenset(x.dropna().astype(str))),
).reset_index()

# 2. Agrupar prefactura por id_atencion → set de códigos CUPS facturados
pf_por_atencion = prefactura.groupby("id_atencion").agg(
    cups_pf_set=("codigo_cups_facturado", lambda x: frozenset(x.dropna().astype(str))),
).reset_index()

sets_atencion = hc_por_atencion.merge(pf_por_atencion, on="id_atencion", how="outer")
sets_atencion["cups_hc_set"] = sets_atencion["cups_hc_set"].apply(
    lambda s: s if isinstance(s, frozenset) else frozenset())
sets_atencion["cups_pf_set"] = sets_atencion["cups_pf_set"].apply(
    lambda s: s if isinstance(s, frozenset) else frozenset())


# 3. Comparar los dos sets DENTRO de cada atención (no fila a fila)
def _comparar(row):
    hc_s, pf_s = row["cups_hc_set"], row["cups_pf_set"]
    con_soporte = pf_s & hc_s
    sin_soporte = pf_s - hc_s
    no_facturados = hc_s - pf_s
    return pd.Series({
        "total_cups_hc": len(hc_s),
        "total_cups_pf": len(pf_s),
        "cups_con_soporte_count": len(con_soporte),
        "cups_sin_soporte_count": len(sin_soporte),
        "cups_no_facturados_count": len(no_facturados),
        "proporcion_cups_con_soporte": (len(con_soporte) / len(pf_s)) if len(pf_s) > 0 else 1.0,
        "proporcion_cups_sin_soporte": (len(sin_soporte) / len(pf_s)) if len(pf_s) > 0 else 0.0,
    })


sets_atencion = pd.concat([sets_atencion, sets_atencion.apply(_comparar, axis=1)], axis=1)
cups_hc_set_by_atencion = dict(zip(sets_atencion["id_atencion"], sets_atencion["cups_hc_set"]))
cups_pf_set_by_atencion = dict(zip(sets_atencion["id_atencion"], sets_atencion["cups_pf_set"]))

print(f"✓ {len(sets_atencion):,} atenciones con set de CUPS (HC y/o PF) construido")
print(f"  - CUPS con soporte (global): {sets_atencion['cups_con_soporte_count'].sum():,}")
print(f"  - CUPS sin soporte (global, glosa potencial): {sets_atencion['cups_sin_soporte_count'].sum():,}")
print(f"  - CUPS con soporte no facturados (global, fuga): {sets_atencion['cups_no_facturados_count'].sum():,}")

✓ 1,201 atenciones con set de CUPS (HC y/o PF) construido
  - CUPS con soporte (global): 2,708.0
  - CUPS sin soporte (global, glosa potencial): 166.0
  - CUPS con soporte no facturados (global, fuga): 252.0


## 2 · Grano = `cruce` (contrato de salida: 1 fila = 1 evaluación HC vs facturación)
`cruce` ya trae, por fila, la pareja `id_detalle_hc` ↔ `id_prefactura` que el
sistema de reglas evaluó (con nulos semánticos: `id_prefactura` nulo =
`NO_FACTURADO`, `id_detalle_hc` nulo = `SIN_SOPORTE_CLINICO`). Esa pareja **sí
es correcta** — no es un artefacto de orden de filas — así que se usa para
traer las columnas de grano fino que 05/07/08 necesitan tal cual
(`codigo_cups`, `codigo_cups_facturado`, `cantidad_realizada`,
`cantidad_facturada`, `valor_unitario`, `valor_total`, `tipo_item`,
`soporte_clinico`, `profesional_responsable`, …).

In [7]:
m = cruce.merge(
    hc[["id_detalle_hc", "id_atencion", "tipo_item", "codigo_cups", "cantidad_realizada",
        "fecha_registro", "soporte_clinico", "profesional_responsable"]],
    on="id_detalle_hc", how="left", suffixes=("", "_hc"),
)
m = m.merge(
    prefactura[["id_prefactura", "codigo_cups_facturado", "descripcion_servicio_facturado",
                "cantidad_facturada", "valor_unitario", "valor_total", "fecha_facturacion"]],
    on="id_prefactura", how="left",
)
assert len(m) == len(cruce), "El merge alteró el número de filas (grano roto)"

if "id_atencion_hc" in m.columns:  # duplicado por el merge con hc, id_atencion de cruce manda
    m = m.drop(columns=["id_atencion_hc"])

m = m.merge(
    atenciones.rename(columns={"eps": "eps_atencion"})[
        ["id_atencion", "id_paciente", "fecha_atencion", "tipo_atencion",
         "diagnostico_principal_cie10", "descripcion_diagnostico", "medico_tratante",
         "sede", "eps_atencion"]
    ],
    on="id_atencion", how="left",
)

m = m.merge(
    pacientes.rename(columns={"eps": "eps_paciente"}),
    on="id_paciente", how="left",
)

print(f"✓ Dataset a grano de cruce: {m.shape[0]:,} filas x {m.shape[1]} columnas "
      f"(relación 1:1 con cruce preservada, igual que antes de este fix)")

✓ Dataset a grano de cruce: 3,126 filas x 34 columnas (relación 1:1 con cruce preservada, igual que antes de este fix)


## 3 · Features derivadas del SET de la atención (paso 3-4 de la lógica corregida)
Aquí está el fix real. En vez de solo mirar si la fila pareada de `cruce`
coincide, cada fila se evalúa contra **todo** el set de CUPS de su atención:

- `codigo_facturado_tiene_soporte_en_atencion`: ¿el código facturado de
  *esta* fila aparece en el set de códigos con soporte clínico de *toda* la
  atención? (antes solo se miraba si coincidía con la única fila de HC
  pareada por `cruce`).
- `codigo_hc_fue_facturado_en_atencion`: ¿el código con soporte clínico de
  *esta* fila aparece en el set de códigos facturados de *toda* la atención?
  (detección de fuga de ingreso a nivel de conjunto).

Estas conviven con las columnas de grano fino (`codigo_cups`,
`codigo_cups_facturado`) que 07 usa para su propia comparación fila a fila
— ambas señales son válidas y se complementan.

In [9]:
m["cups_hc_set_atencion"] = m["id_atencion"].map(cups_hc_set_by_atencion)
m["cups_pf_set_atencion"] = m["id_atencion"].map(cups_pf_set_by_atencion)

m["codigo_facturado_tiene_soporte_en_atencion"] = m.apply(
    lambda r: int(pd.notna(r["codigo_cups_facturado"])
                  and r["codigo_cups_facturado"] in r["cups_hc_set_atencion"]),
    axis=1,
)
m["codigo_hc_fue_facturado_en_atencion"] = m.apply(
    lambda r: int(pd.notna(r["codigo_cups"])
                  and r["codigo_cups"] in r["cups_pf_set_atencion"]),
    axis=1,
)

m = m.merge(
    sets_atencion[["id_atencion", "total_cups_hc", "total_cups_pf", "cups_con_soporte_count",
                   "cups_sin_soporte_count", "cups_no_facturados_count",
                   "proporcion_cups_con_soporte", "proporcion_cups_sin_soporte"]],
    on="id_atencion", how="left",
)
m = m.drop(columns=["cups_hc_set_atencion", "cups_pf_set_atencion"])

print("✓ Features de comparación por set calculadas")
print(f"  - Filas con soporte confirmado por set: {m['codigo_facturado_tiene_soporte_en_atencion'].sum():,}")
print(f"  - Filas con facturación confirmada por set: {m['codigo_hc_fue_facturado_en_atencion'].sum():,}")

✓ Features de comparación por set calculadas
  - Filas con soporte confirmado por set: 2,805
  - Filas con facturación confirmada por set: 2,805


## 4 · Features específicas por tipo de atención (reglas de negocio del jefe)
Usa `tipo_atencion` traído de `atenciones` (paso 4 de la lógica corregida).

In [11]:
m["es_ambulatorio"] = (m["tipo_atencion"] == "Ambulatoria").astype(int)
m["es_hospitalario"] = (m["tipo_atencion"] == "Hospitalizacion").astype(int)
m["es_urgencia"] = (m["tipo_atencion"] == "Urgencias").astype(int)

valor_atencion = prefactura.groupby("id_atencion")["valor_total"].sum().rename("valor_total_atencion")
m = m.merge(valor_atencion, on="id_atencion", how="left")

# Ambulatorio: servicios de alto valor requieren autorización EPS
m["servicio_alto_valor_ambulatorio"] = (
    (m["es_ambulatorio"] == 1) & (m["valor_total_atencion"].fillna(0) > 100000)
).astype(int)

# Hospitalario: tratamientos complejos requieren más soporte médico diario
m["tratamiento_complejo_hospitalario"] = (
    (m["es_hospitalario"] == 1) & (m["valor_total_atencion"].fillna(0) > 200000)
).astype(int)

m["dias_atencion_a_facturacion"] = (m["fecha_facturacion"] - m["fecha_atencion"]).dt.days
m["facturacion_tardia"] = (m["dias_atencion_a_facturacion"] > 7).fillna(False).astype(int)

print("✓ Features por tipo de atención calculadas")
print(f"  - Ambulatorio: {m['es_ambulatorio'].sum():,} | Hospitalario: {m['es_hospitalario'].sum():,} "
      f"| Urgencia: {m['es_urgencia'].sum():,}")

✓ Features por tipo de atención calculadas
  - Ambulatorio: 1,067 | Hospitalario: 970 | Urgencia: 1,089


## 5 · Target y features finales
`target` (0/1) y `resultado` (CONSISTENTE/INCONSISTENTE) — el contrato exige
mantener **ambas** versiones: 05/07 usan `target`, 08 usa `resultado`.

In [13]:
m["target"] = (m["resultado"] == "INCONSISTENTE").astype(int)
m["mes_atencion"] = m["fecha_atencion"].dt.month

bins = [-1, 17, 39, 59, 79, 200]
labels = ["0-17", "18-39", "40-59", "60-79", "80+"]
m["grupo_etario"] = pd.cut(m["edad"], bins=bins, labels=labels).astype(str)

# Rellenos donde el merge quedó vacío por falta de contraparte (NO_FACTURADO / SIN_SOPORTE_CLINICO)
m["cantidad_realizada"] = m["cantidad_realizada"].fillna(0)
m["cantidad_facturada"] = m["cantidad_facturada"].fillna(0)
m["valor_unitario"] = m["valor_unitario"].fillna(0)
m["valor_total"] = m["valor_total"].fillna(0)
m["soporte_clinico"] = m["soporte_clinico"].fillna("NO")
m["tipo_item"] = m["tipo_item"].fillna("SIN_DATO")
m["profesional_responsable"] = m["profesional_responsable"].fillna("SIN_DATO")

m["eps_coincide_paciente"] = np.where(
    m["eps_atencion"].notna() & m["eps_paciente"].notna(),
    (m["eps_atencion"] == m["eps_paciente"]).astype(float),
    np.nan,
)

print(f"✓ Target: {m['target'].value_counts().to_dict()} ({m['target'].mean()*100:.1f}% INCONSISTENTE)")
print(f"✓ Dataset final: {m.shape[0]:,} filas x {m.shape[1]} columnas")

✓ Target: {0: 2477, 1: 649} (20.8% INCONSISTENTE)
✓ Dataset final: 3,126 filas x 55 columnas


## 6 · Contrato de columnas: qué entra al modelo y qué no
Se valida explícitamente contra lo que **05, 07 y 08 leen del dataset**, para
que un cambio futuro en este notebook no pueda romperlos en silencio.

In [15]:
FEATURES_NUM = [
    # Features demográficas y de cantidades/valores (grano fino, igual que antes)
    "edad", "cantidad_realizada", "cantidad_facturada", "valor_unitario", "valor_total",
    "mes_atencion", "dias_atencion_a_facturacion",
    # Features de comparación por SET (lógica corregida)
    "codigo_facturado_tiene_soporte_en_atencion", "codigo_hc_fue_facturado_en_atencion",
    "total_cups_hc", "total_cups_pf", "cups_con_soporte_count", "cups_sin_soporte_count",
    "cups_no_facturados_count", "proporcion_cups_con_soporte", "proporcion_cups_sin_soporte",
    # Features de contexto y reglas de negocio por tipo de atención
    "es_ambulatorio", "es_hospitalario", "es_urgencia",
    "servicio_alto_valor_ambulatorio", "tratamiento_complejo_hospitalario", "facturacion_tardia",
    # Consistencia
    "eps_coincide_paciente",
]

FEATURES_CAT = [
    "sexo", "tipo_documento", "tipo_afiliacion", "ciudad",
    "eps_atencion", "tipo_atencion", "sede", "tipo_item", "soporte_clinico", "grupo_etario",
    "diagnostico_principal_cie10", "medico_tratante", "profesional_responsable",
]

faltan = [c for c in FEATURES_NUM + FEATURES_CAT if c not in m.columns]
assert not faltan, f"Faltan columnas del contrato propio: {faltan}"

# --- Verificación cruzada contra lo que 05 / 07 / 08 leen textualmente del maestro ---
REQUERIDAS_08 = ["edad", "cantidad_realizada", "cantidad_facturada", "valor_unitario", "valor_total",
                  "mes_atencion", "sexo", "eps_atencion", "tipo_afiliacion", "ciudad", "tipo_documento",
                  "tipo_atencion", "sede", "tipo_item", "soporte_clinico", "grupo_etario",
                  "diagnostico_principal_cie10", "medico_tratante", "profesional_responsable", "resultado"]
REQUERIDAS_07 = ["codigo_cups_facturado", "codigo_cups", "cantidad_facturada", "cantidad_realizada",
                  "soporte_clinico", "fecha_facturacion", "fecha_atencion", "valor_unitario", "valor_total",
                  "edad", "tipo_atencion", "sede", "eps_atencion", "tipo_afiliacion", "tipo_item",
                  "grupo_etario", "sexo", "diagnostico_principal_cie10", "resultado"]
faltan08 = [c for c in REQUERIDAS_08 if c not in m.columns]
faltan07 = [c for c in REQUERIDAS_07 if c not in m.columns]
assert not faltan08, f"Faltan columnas requeridas por 08_modelo_cnn_transfer_learning: {faltan08}"
assert not faltan07, f"Faltan columnas requeridas por 07_entrenamiento_xgboost_avanzado: {faltan07}"

doc = {
    "target": {"columna": "target", "definicion": "1 si resultado == INCONSISTENTE"},
    "features_numericas": FEATURES_NUM,
    "features_categoricas": FEATURES_CAT,
    "excluidas_por_leakage": {
        "columnas": ["resultado", "tipo_alerta", "severidad", "descripcion_alerta"],
        "motivo": "Salidas del sistema de reglas que generó el target; solo uso explicativo.",
    },
    "excluidas_ids": ["id_cruce", "id_atencion", "id_prefactura", "id_detalle_hc", "id_paciente"],
    "excluidas_texto_libre_y_fechas_crudas": [
        "descripcion_servicio_facturado", "descripcion_diagnostico",
        "fecha_facturacion", "fecha_registro", "fecha_atencion",
    ],
    "notas": [
        "Grano: 1 fila = 1 registro de cruce (idéntico al de antes de este fix; compatible con 05/07/08).",
        "Lógica corregida: las features de comparación se calculan agrupando por atención (set de CUPS "
        "en HC vs set de CUPS facturados), no solo con la fila pareada de cruce.",
        "codigo_facturado_tiene_soporte_en_atencion: 1 si el codigo_cups_facturado de esta fila aparece "
        "en el set de códigos con soporte clínico de TODA la atención.",
        "codigo_hc_fue_facturado_en_atencion: 1 si el codigo_cups con soporte de esta fila aparece en el "
        "set de códigos facturados de TODA la atención (detección de fuga de ingreso).",
        "07 y 08 calculan sus propias features fila a fila (p.ej. coincide_codigo_cups) usando "
        "codigo_cups y codigo_cups_facturado de esta misma fila; válido porque las FK de cruce "
        "(id_detalle_hc/id_prefactura) sí son la pareja correcta (ver verificación en la celda de cierre).",
    ],
}

print("✓ Contrato con 05 / 07 / 08 verificado: todas las columnas requeridas están presentes")

✓ Contrato con 05 / 07 / 08 verificado: todas las columnas requeridas están presentes


## 7 · Exportar dataset maestro

In [17]:
to_csv_seguro(m, DATA_PROC / "dataset_maestro.csv", index=False)
write_text_seguro(OUT_REP / "columnas_modelado.json",
                   json.dumps(doc, ensure_ascii=False, indent=2))

print(f"✓ dataset_maestro.csv: {m.shape[0]:,} x {m.shape[1]} -> {DATA_PROC}")
print(f"✓ columnas_modelado.json -> {OUT_REP / 'columnas_modelado.json'}")

✓ dataset_maestro.csv: 3,126 x 55 -> /home/claude/LINE/data/processed
✓ columnas_modelado.json -> /home/claude/LINE/outputs/reports/columnas_modelado.json


## ✅ Verificación de cierre
Chequeos ejecutables (no solo una lista en markdown): existencia de salidas,
grano preservado, y que la pareja `id_detalle_hc`/`id_prefactura` de `cruce`
efectivamente sustenta la comparación fila a fila que hacen 07/08.

In [19]:
assert (DATA_PROC / "dataset_maestro.csv").exists()
assert (OUT_REP / "columnas_modelado.json").exists()

chequeo = pd.read_csv(DATA_PROC / "dataset_maestro.csv")
assert len(chequeo) == len(cruce), "El maestro exportado no preserva el grano de cruce"
assert {"target", "resultado"}.issubset(chequeo.columns)
assert set(chequeo["resultado"].unique()) <= {"CONSISTENTE", "INCONSISTENTE"}

# La pareja de cruce (id_detalle_hc / id_prefactura) sustenta la comparación fila a fila de 07/08:
# donde CODIGO_NO_COINCIDE, los códigos de la fila pareada deben diferir SIEMPRE.
ambos = chequeo.dropna(subset=["codigo_cups", "codigo_cups_facturado"])
coincide_fila = ambos["codigo_cups"] == ambos["codigo_cups_facturado"]
crosstab_chk = pd.crosstab(ambos["tipo_alerta"], coincide_fila)
assert (crosstab_chk.loc["CODIGO_NO_COINCIDE"].get(True, 0) == 0), (
    "La pareja de cruce no es confiable para la comparación fila a fila")

print(f"✓ dataset_maestro.csv: {len(chequeo):,} filas (= {len(cruce):,} de cruce, grano preservado)")
print(f"✓ columnas_modelado.json: {OUT_REP / 'columnas_modelado.json'}")
print("✓ Pareja id_detalle_hc/id_prefactura de cruce verificada como confiable para comparación fila a fila")
print()
print("Lógica corregida aplicada en esta versión:")
print("  - Grano preservado (1 fila = 1 registro de cruce) -> 05/07/08 no requieren cambios")
print("  - Comparación de sets de CUPS por atención, no solo por la fila pareada")
print("  - Features específicas por tipo de atención (ambulatorio/hospitalario/urgencia)")
print("  - target (0/1) y resultado (CONSISTENTE/INCONSISTENTE) exportados, como exige el contrato")

✓ dataset_maestro.csv: 3,126 filas (= 3,126 de cruce, grano preservado)
✓ columnas_modelado.json: /home/claude/LINE/outputs/reports/columnas_modelado.json
✓ Pareja id_detalle_hc/id_prefactura de cruce verificada como confiable para comparación fila a fila

Lógica corregida aplicada en esta versión:
  - Grano preservado (1 fila = 1 registro de cruce) -> 05/07/08 no requieren cambios
  - Comparación de sets de CUPS por atención, no solo por la fila pareada
  - Features específicas por tipo de atención (ambulatorio/hospitalario/urgencia)
  - target (0/1) y resultado (CONSISTENTE/INCONSISTENTE) exportados, como exige el contrato
